# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets (referenced via '@id')
print("Available record sets:")
for rs in metadata.record_sets:
    print(f"- {rs['@id']} : {rs.get('name', 'No name')}")

# For each record set, print the available fields and columns (@id)
for rs in metadata.record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - {field['@id']}: {field.get('name', 'No name')} ({field.get('data_type', 'No type')})")
    if 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            print(f"    - {col['@id']}: {col.get('name', 'No name')} ({col.get('data_type', 'No type')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set: {record_set_id}")

if record_sets:
    first_record_set_id = record_sets[0]
    print(f"\nColumns in first record set ({first_record_set_id}):")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For this demo, operate on the first record set found
import numpy as np
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]

    # Identify numeric fields (try to guess)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields in {record_set_id}: {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()  # Use mean as threshold for demo
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a group field (categorical, e.g. string/object)
        group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Visualize numeric field distribution for the primary record set
if record_sets and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If there's a categorical group field with not too many levels, make a boxplot
    if group_field and df[group_field].nunique() <= 10:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we've loaded and explored the FAIR² dataset on mass spectrometry analysis of human extracellular vesicles. After loading the dataset using `mlcroissant`, we reviewed the available record sets and fields, extracted data into DataFrames, and performed basic exploratory steps such as filtering, normalization, grouping, and visualization. Depending on your research goals, you can now proceed with further, domain-specific analyses. For more detail on data provenance and semantics, consult the Croissant schema and documentation.